In [30]:
%matplotlib inline
import sys
import os
sys.path.append(os.path.abspath(".."))
import importlib

from functions import color_palettes
importlib.reload(color_palettes)
from functions.color_palettes import *

from functions import utils
importlib.reload(utils)
from functions.utils import *

from functions import config
importlib.reload(config)
from functions.config import *

In [3]:
gemini_results_path=r"C:\Users\aoezek\Desktop\master_thesis\master_thesis\gemini_validation\validation_results.xlsx"

ground_truth_csv_file = r"C:\Users\aoezek\Desktop\jfb_data\food-scan-benchmark-dataset\food_scan_bench_v1.csv"
image_folder = Path(r"C:\Users\aoezek\Desktop\Thesis\gemini_validation\gemini_validation_january_ai_data")

df = pd.read_csv(ground_truth_csv_file)
selected_images = {f.name for f in image_folder.glob("*.jpg")}

filtered_df = df[df["image_filename"].isin(selected_images)]

filtered_df.to_excel(r"C:\Users\aoezek\Desktop\Thesis\gemini_validation\filtered_ground_truth.xlsx", index=False)

In [4]:
ground_truth=filtered_df
gemini_results=pd.read_excel(gemini_results_path)

In [5]:
gemini_results.rename(columns={"image_name": "image_filename",
                               "carbs_g": "total_carbs",
                                "protein_g": "total_protein",
                                "fat_g": "total_fat",
                               }, inplace=True)

In [6]:
filename_col = "image_filename"

nutrition_cols = [
    "total_calories",
    "total_carbs",
    "total_fat",
    "total_protein"
]

df = ground_truth.merge(
    gemini_results,
    on=filename_col,
    suffixes=("_ground_truth", "_prediction")
)

In [7]:
df.drop(columns=['image_id', 'ingredients_list', 'ingredient_names', 'num_ingredients'], inplace=True)

---
#### Validation Metrics
---

In [ ]:
results = []

for col in nutrition_cols:

    y_true = df[f"{col}_ground_truth"]
    y_pred = df[f"{col}_prediction"]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

#mape might return undefined if there are 0 enties

    corr = y_true.corr(y_pred)
    results.append({
        "metric": col,
        "MAE": mae,
        "RMSE": rmse,
    })

results_df = pd.DataFrame(results)

print(results_df)

In [ ]:
results_2 = []
for col in nutrition_cols:

    y_true = df[f"{col}_ground_truth"]
    y_pred = df[f"{col}_prediction"]
    ape = np.abs((y_pred - y_true) / y_true) * 100
    within_10 = (ape <= 10).mean() * 100
    within_25 = (ape <= 25).mean() * 100
    within_50 = (ape <= 50).mean() * 100

    over_50 = (ape > 50).mean() * 100
    over_100 = (ape > 100).mean() * 100

    median_error = np.median(ape)

    results_2.append({
        "metric": col,
        "Median_APE_%": median_error,
        "Within_10%": within_10,
        "Within_25%": within_25,
        "Within_50%": within_50,
        "Over_50%": over_50,
        "Over_100%": over_100
    })

results_df = pd.DataFrame(results_2)

print(results_df)

---
#### VALIDATION ON THE MANUELLY LABELLED CROSS CHECK BATCH OF THE ORIGINAL DATA
---

In [49]:
ground_truth_manuel = r"C:\Users\aoezek\Desktop\master_thesis\master_thesis\gemini_validation\ground_truth_manuel_cross_check.xlsx"
gemini_labels = r"C:\Users\aoezek\Desktop\master_thesis\master_thesis\gemini_validation\cross_check_images_gemini.xlsx"

df_ground_truth = pd.read_excel(ground_truth_manuel)
df_gemini_labels = pd.read_excel(gemini_labels)


In [50]:
comparison_cols = [
    "carbs_g",
    "protein_g",
    "fat_g",
    "estimated_total_weight_g",
    "total_calories",
]

In [51]:
df_compare = df_ground_truth.merge(
    df_gemini_labels,
    on='image_name',
    suffixes=("_ground_truth", "_prediction")
)

In [54]:
def wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

In [ ]:
results = []

for col in comparison_cols:
    y_true = df_compare[f"{col}_ground_truth"]
    y_pred = df_compare[f"{col}_prediction"]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    corr = y_true.corr(y_pred)
    wm = wmape(y_true, y_pred)

    results.append({
        "metric": col,
        "mean": y_true.mean(),
        "MAE": mae,
        "RMSE":rmse,
        "WMAPE":wm,
        "Corr": corr,
    })

results_df = pd.DataFrame(results)
print(results_df)
results_df.to_latex("manual_validation_results_table.tex", index=False)

In [ ]:
for col in comparison_cols:
    y_true = df_compare[f"{col}_ground_truth"]
    y_pred = df_compare[f"{col}_prediction"]

    mean_val = y_true.mean()
    mae_val = np.abs(y_pred - y_true).mean()

    print(f"{col}")
    print(f"Mean ground truth: {mean_val:.1f}")
    print(f"MAE: {mae_val:.1f} ({mae_val/mean_val*100:.1f}% of mean)")
    print()

---
#### LABELING VALIDATION JANUARY AI DATASE
---

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
def label_correct(pred, truth, threshold=0.5):
    # cosine similarity
    vecs = model.encode([str(pred), str(truth)])
    cos_score = cosine_similarity([vecs[0]], [vecs[1]])[0][0]
    return cos_score >= threshold

In [42]:
df["label_correct"] = df.apply(
    lambda row: label_correct(row["meal_label"], row["meal_name"]), axis=1
)

In [ ]:
for _, row in df.iterrows():
    pred = row["meal_label"]
    truth = row["meal_name"]
    correct = row["label_correct"]
    
    status = "True" if correct else "False"
    print(f"{status} PRED: {pred}")
    print(f"TRUTH: {truth}")
    print()

print(f"─" * 60)
print(f"Label Accuracy: {df['label_correct'].mean()*100:.1f}%  ({df['label_correct'].sum()}/{len(df)})")

In [ ]:
df["similarity"] = df.apply(
    lambda row: cosine_similarity(
        model.encode([str(row["meal_label"]), str(row["meal_name"])])[[0]], 
        model.encode([str(row["meal_label"]), str(row["meal_name"])])[[1]]
    )[0][0], axis=1
)

print(f"Mean Similarity: {df['similarity'].mean():.3f}")
print(f"Median Similarity: {df['similarity'].median():.3f}")

In [ ]:
with open("label_validation_results_table.tex", "w") as f:
    f.write(r"\begin{table}[h]" + "\n")
    f.write(r"\centering" + "\n")
    f.write(r"\caption{Meal Label Prediction Results}" + "\n")
    f.write(r"\begin{tabular}{p{7cm}p{4.5cm}c}" + "\n")
    f.write(r"\hline" + "\n")
    f.write(r"\textbf{Prediction} & \textbf{Ground Truth} & \textbf{Correct} \\" + "\n")
    f.write(r"\hline" + "\n")

    for _, row in df.iterrows():
        pred = row["meal_label"].replace("&", r"\&")
        truth = row["meal_name"].replace("&", r"\&")
        correct = r"\checkmark" if row["label_correct"] else r"\texttimes"
        f.write(f"{pred} & {truth} & {correct} \\\\\n")

    acc = df['label_correct'].mean() * 100
    total = len(df)
    correct_n = df['label_correct'].sum()

    f.write(r"\hline" + "\n")
    f.write(f"\\multicolumn{{2}}{{l}}{{\\textbf{{Label Accuracy}}}} & \\textbf{{{acc:.1f}\\%}} \\\\\n")
    f.write(r"\hline" + "\n")
    f.write(r"\end{tabular}" + "\n")
    f.write(r"\label{tab:meal_label_results}" + "\n")
    f.write(r"\end{table}" + "\n")